# Setup

In [1]:
!pip install wandb --quiet

import wandb
import warnings
from kaggle_secrets import UserSecretsClient
warnings.filterwarnings("ignore")

user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [2]:
run = wandb.init(
    entity="23f2002974-dl-genai-project",
    project="23f2002974-t12026",
    name="AST_Pretrained_Run_f1",
    config={
        "architecture": "AST",
        "pretrained_model": "MIT/ast-finetuned-audioset-10-10-0.4593",
        "epochs": 16,
    }
)

# Libraries and imports

In [3]:
import os
import random

import numpy as np
import pandas as pd
from tqdm import tqdm

import librosa
import torchaudio.transforms as T

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from transformers import ASTConfig, ASTForAudioClassification, ASTFeatureExtractor

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# Data Pre-processing & Feature Extraction

In [23]:
"""Config"""
# Standard audio settings
SR = 22050
N_MELS = 128
DURATION = 30 

# Where the data is coming from
GENRE_INPUT = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
NOISE_INPUT = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"

# Where we want to save the processed files
GENRE_OUTPUT = "/kaggle/working/processed_mels"
NOISE_OUTPUT = "/kaggle/working/processed_noise_mels"

STEM_FILES = ["drums.wav", "bass.wav", "vocals.wav", "other.wav"]

In [41]:
def preprocess_dataset():
    
    '''Create the main output directory'''
    os.makedirs(GENRE_OUTPUT, exist_ok=True)

    all_items = os.listdir(GENRE_INPUT)
    genres = []

    for item in all_items:
        if os.path.isdir(os.path.join(GENRE_INPUT, item)):
            genres.append(item)

    for genre in genres:
        genre_input_path = os.path.join(GENRE_INPUT, genre)
        genre_output_path = os.path.join(GENRE_OUTPUT, genre)
        os.makedirs(genre_output_path, exist_ok=True)

        '''Get list of songs in the genre folder'''
        all_songs = os.listdir(genre_input_path)
        songs = []

        for s in all_songs:
            if os.path.isdir(os.path.join(genre_input_path, s)):
                songs.append(s)

        for song in tqdm(songs, desc=f"Processing {genre}"):

            song_input = os.path.join(genre_input_path, song)
            song_output = os.path.join(genre_output_path, song)
            os.makedirs(song_output, exist_ok=True)

            for stem in STEM_FILES:

                input_file = os.path.join(song_input, stem)
                output_file = os.path.join(song_output, stem.replace(".wav", ".npy"))

                '''Load audio'''

                audio, _ = librosa.load(input_file, sr=SR, duration=DURATION)

                '''Compute Mel Spectrogram'''

                mel = librosa.feature.melspectrogram(
                    y=audio,
                    sr=SR,
                    n_mels=N_MELS,
                    n_fft=2048,
                    hop_length=512
                )

                '''Save the Mel Spectrogram'''

                np.save(output_file, mel.astype(np.float32))

In [42]:
preprocess_dataset()

Processing pop: 100%|██████████| 100/100 [00:00<00:00, 19864.10it/s]


In [43]:
def preprocess_noise():
    
    '''Create the output folder using your defined NOISE_OUTPUT'''
    os.makedirs(NOISE_OUTPUT, exist_ok=True)

    '''Get all wav files from NOISE_INPUT'''
    noise_files = []
    all_files = os.listdir(NOISE_INPUT)

    for f in all_files:
        if f.endswith(".wav"):
            noise_files.append(f)

    print(f"Processing {len(noise_files)} noise files")

    for file in tqdm(noise_files):

        input_path = os.path.join(NOISE_INPUT, file)
        output_path = os.path.join(NOISE_OUTPUT, file.replace(".wav", ".npy"))

        '''Load audio using your SR'''

        audio, _ = librosa.load(input_path, sr=SR)

        '''Convert to Mel Spectrogram using your N_MELS'''

        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=SR,
            n_mels=N_MELS,
            n_fft=2048,
            hop_length=512
        )

        '''Save the Mel Spectrogram'''

        np.save(output_path, mel.astype(np.float32))

In [44]:
preprocess_noise()

Processing 2000 noise files


100%|██████████| 2000/2000 [00:00<00:00, 137561.01it/s]


# Custom Dataset :

In [45]:
class Dataset(Dataset):

    def __init__(self, processed_dir, noise_dir, song_dict, genres, size=30000, is_val=False):

        self.processed_dir = processed_dir
        self.song_dict = song_dict
        self.genres = genres
        self.size = size
        self.is_val = is_val

        # genre to index
        self.genre_to_idx = {}
        for i in range(len(genres)):
            self.genre_to_idx[genres[i]] = i

        # load noise file paths
        self.noise_files = []
        files = os.listdir(noise_dir)

        for f in files:
            if f.endswith(".npy"):
                full_path = os.path.join(noise_dir, f)
                self.noise_files.append(full_path)

        # AST input size
        self.target_h = 128
        self.target_w = 1024

        # spec augment
        self.freq_mask = T.FrequencyMasking(freq_mask_param=25)
        self.time_mask = T.TimeMasking(time_mask_param=50)


    def __len__(self):
        return self.size


    def __getitem__(self, idx):

        # balanced genre selection
        genre = self.genres[idx % len(self.genres)]
        songs = self.song_dict[genre]

        # empty mel
        mixed_mel = np.zeros((128,1292), dtype=np.float32)

        stems = ["drums.npy", "bass.npy", "vocals.npy", "other.npy"]

        for stem in stems:

            song_id = random.choice(songs)

            path = os.path.join(self.processed_dir, genre, song_id, stem)

            mel = np.load(path)

            # fix width
            if mel.shape[1] > 1292:
                mel = mel[:, :1292]
            else:
                pad = 1292 - mel.shape[1]
                mel = np.pad(mel, ((0,0),(0,pad)))

            
            gain = random.uniform(0.6,1.4)

            mixed_mel = mixed_mel + mel * gain


        # add noise
        if not self.is_val:

            num_noise = random.randint(1,3)

            for i in range(num_noise):

                noise_file = random.choice(self.noise_files)

                noise = np.load(noise_file)

                if mixed_mel.shape[1] > noise.shape[1]:

                    start = random.randint(
                        0,
                        mixed_mel.shape[1] - noise.shape[1]
                    )

                    end = start + noise.shape[1]

                    mixed_mel[:, start:end] = mixed_mel[:, start:end] + \
                        noise * random.uniform(0.05,0.2)


        # convert to db
        mel_db = librosa.power_to_db(np.maximum(mixed_mel,1e-10))

        # normalize
        mean = mel_db.mean()
        std = mel_db.std()

        mel_db = (mel_db - mean) / (std + 1e-6)


        # transpose for AST
        mel = mel_db.T


        # crop
        if self.is_val:
            start = (mel.shape[0] - self.target_w) // 2
        else:
            start = random.randint(0, mel.shape[0] - self.target_w)

        end = start + self.target_w

        mel = mel[start:end, :]

        mel = torch.tensor(mel, dtype=torch.float32)


        # spec augment
        if not self.is_val:

            mel = mel.unsqueeze(0)

            mel = self.freq_mask(mel)

            mel = self.time_mask(mel)

            mel = mel.squeeze(0)


        label = self.genre_to_idx[genre]

        return mel, label

# Train / Test Split

In [46]:
def stratified_split(root_dir, val_ratio=0.15):

    train_songs = {}
    val_songs = {}

    # get genre folders
    genres = []

    for d in os.listdir(root_dir):
        path = os.path.join(root_dir, d)

        if os.path.isdir(path):
            genres.append(d)

    genres.sort()

    for genre in genres:

        genre_path = os.path.join(root_dir, genre)

        songs = []

        for s in os.listdir(genre_path):

            song_path = os.path.join(genre_path, s)

            if os.path.isdir(song_path):
                songs.append(s)

        songs.sort()
        random.shuffle(songs)

        # split into train and validation
        split = int(len(songs) * val_ratio)

        val_songs[genre] = songs[:split]
        train_songs[genre] = songs[split:]

        print("Genre", genre.ljust(10), "| Train:", len(train_songs[genre]), "| Val:", len(val_songs[genre]))

    return train_songs, val_songs, genres

In [47]:
train_songs, val_songs, genres = stratified_split(GENRE_OUTPUT)

Genre blues      | Train: 85 | Val: 15
Genre classical  | Train: 85 | Val: 15
Genre country    | Train: 85 | Val: 15
Genre disco      | Train: 85 | Val: 15
Genre hiphop     | Train: 85 | Val: 15
Genre jazz       | Train: 85 | Val: 15
Genre metal      | Train: 85 | Val: 15
Genre pop        | Train: 85 | Val: 15
Genre reggae     | Train: 85 | Val: 15
Genre rock       | Train: 85 | Val: 15


# Data Loaders

In [48]:
train_dataset = Dataset(GENRE_OUTPUT, NOISE_OUTPUT, train_songs, genres, size=20000)
val_dataset = Dataset(GENRE_OUTPUT, NOISE_OUTPUT, val_songs, genres, size=2000, is_val=True)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

# Model Architecture

## AST Pretrained

In [49]:
model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = ASTFeatureExtractor.from_pretrained(model_name)

config = ASTConfig.from_pretrained(model_name)
config.num_labels = 10
model = ASTForAudioClassification.from_pretrained(
    model_name, 
    config=config, 
    ignore_mismatched_sizes=True
)

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [50]:
model = model.to(device)

# Loss Function, Optimizer, and Metrics

In [51]:
optimizer = torch.optim.AdamW(model.parameters(),lr=5e-5,weight_decay=5e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=18)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Training Loop definition

In [52]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    pbar = tqdm(loader)

    for x, y in pbar:

        # move data to device
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        # backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

        pbar.set_description(
            f"Train Loss {total_loss/(total/loader.batch_size):.4f} Acc {100*correct/total:.2f}%"
        )

    avg_loss = total_loss / len(loader)
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1

# Validation Loop definition

In [53]:
def validate_one_epoch(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x, y in tqdm(loader):

            x = x.to(device)
            y = y.to(device)

            logits = model(x).logits

            loss = criterion(logits, y)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1

# Model training and WndB logging

In [54]:
def train_model(model,train_loader,val_loader,criterion,optimizer,scheduler,device,epochs,patience):
    best_val_acc = 0
    patience_counter = 0

    for epoch in range(epochs):

        print(f"\nEpoch {epoch+1}/{epochs}")

        # -------- TRAIN --------
        train_loss, train_acc, train_f1 = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        # -------- VALIDATE --------
        val_loss, val_acc, val_f1 = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device
        )

        print(
            f"\nEpoch [{epoch+1}/{epochs}] "
            f"| Train Loss: {train_loss:.4f} "
            f"| Train Acc: {train_acc:.4f} "
            f"| Train F1: {train_f1:.4f} "
            f"| Val Loss: {val_loss:.4f} "
            f"| Val Acc: {val_acc:.4f} "
            f"| Val F1: {val_f1:.4f}"
        )

        # -------- SCHEDULER --------
        scheduler.step(val_acc)

        #-------- WANDB LOG --------
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
            "lr": optimizer.param_groups[0]["lr"]
        })

        # -------- EARLY STOPPING --------
        if val_acc > best_val_acc:

            best_val_acc = val_acc
            patience_counter = 0

            torch.save(model.state_dict(), "best_model.pth")

            print("Model Improved. Saved.")

        else:

            patience_counter += 1

            print(f"Early stopping counter: {patience_counter}/{patience}")

            if patience_counter >= patience:

                print("Early stopping triggered")
                break

    print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")

# Training execution¶

In [55]:
epochs= 18
patience= 5

In [ ]:
# train_model(model=model,
#         train_loader=train_loader,
#         val_loader=val_loader,
#         criterion=criterion,
#         optimizer=optimizer,
#         scheduler=scheduler,
#         device=device,
#         epochs=epochs,
#         patience=patience
#     )

# Model upload to kagglehub

In [57]:
import kagglehub
model_path = "/kaggle/working/pretrained_ast.pth"


torch.save(model.state_dict(), model_path)

print("Model saved locally at:", model_path)

Model saved locally at: /kaggle/working/pretrained_ast.pth


In [6]:
KAGGLE_USERNAME = "uditmaurya1588"
MODEL = "Dlgenai"             
FRAMEWORK = "pytorch"                  
VARIATION = "ast-v1"              

handle = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"

In [7]:
kagglehub.model_upload(
    handle,
    model_path,
    version_notes="Initial model version"
)

Uploading Model https://api.kaggle.com/models/uditmaurya1588/Dlgenai/pytorch/ast-v1 ...
Starting upload for file /kaggle/input/models/uditmaurya1588/ast-pretrianed/pytorch/default/1/Ast_0.93.pth


Uploading: 100%|██████████| 345M/345M [00:03<00:00, 93.0MB/s] 

Upload successful: /kaggle/input/models/uditmaurya1588/ast-pretrianed/pytorch/default/1/Ast_0.93.pth (329MB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/uditmaurya1588/Dlgenai/pytorch/ast-v1
